# ROHO: private Kaggle smoke test
## Goal
Run a small exploratory harness-optimization pilot with a frozen local Qwen model. This is not a completed paper experiment. Keep this notebook and the source dataset private. No paid API key is needed.
## Setup
Attach your private dataset containing `roho-source.zip`, enable an available GPU, and enable Internet. Check your displayed quota. Source: [Kaggle notebook documentation](https://www.kaggle.com/docs/notebooks).

In [ ]:
from pathlib import Path
import os, zipfile, subprocess, sys, shutil
source_archives = list(Path('/kaggle/input').rglob('roho-source.zip'))
assert len(source_archives) == 1, 'Attach exactly one private dataset containing roho-source.zip'
workspace = Path('/kaggle/working')
project = workspace / 'roho'
if not project.exists():
    with zipfile.ZipFile(source_archives[0]) as archive:
        for member in archive.infolist():
            target = (workspace / member.filename).resolve()
            assert target.is_relative_to(project.resolve()), 'Unsafe source archive path'
            assert (member.external_attr >> 16) & 0o170000 != 0o120000, 'Symlinks not allowed'
        archive.extractall(workspace)
else:
    print('Keeping existing source directory. Do not change source while resuming a run.')
os.chdir(project)
print('Project:', project)
def command(*args):
    subprocess.run([sys.executable, *args], cwd=project, check=True)


### Install dependencies and run software checks
Retain Kaggle's CUDA-enabled PyTorch. Unit tests do not prove learning. The final check downloads the configured model into Kaggle's cache and requires one sane JSON generation before an experimental output directory is created. If dependencies were already imported, restart the kernel after installing before starting the experiment.

In [ ]:
command('-m', 'pip', 'install', '-r', 'requirements-kaggle.txt')
command('-m', 'unittest', 'discover', '-s', 'tests', '-v')
command('-m', 'roho', 'doctor')
import json, torch
assert torch.cuda.is_available(), 'Enable GPU before proceeding'
print(torch.cuda.get_device_name(0))
smoke_config = json.loads((project / 'configs/kaggle-smoke.json').read_text())
probe_directory = Path('/kaggle/working/roho-generation-probe')
if probe_directory.exists():
    shutil.rmtree(probe_directory)
probe_code = r'''
from pathlib import Path
import sys
from roho.backend import HFBackend
from roho.common import parse_object
root = Path(sys.argv[1])
root.mkdir(parents=True)
backend = HFBackend(root, sys.argv[2], sys.argv[3], 8192, budget=10000)
text, count = backend.generate([
    {'role': 'system', 'content': 'Return a short JSON object with key ok and value true.'},
    {'role': 'user', 'content': 'Respond now.'},
], 48, 20260920, 0)
print('PREFLIGHT_OUTPUT', repr(text), 'TOKENS', count, flush=True)
if parse_object(text).get('ok') is not True:
    raise RuntimeError('Generation-integrity preflight failed')
'''
subprocess.run([sys.executable, '-c', probe_code, str(probe_directory), smoke_config['model_id'], smoke_config['revision']], cwd=project, check=True)
shutil.rmtree(probe_directory)
print('Generation-integrity preflight passed.')


## Steps
### Run real Qwen smoke optimization
Five train and five dev tasks; A/C/D/E; one round; one proposal per learned arm. The adapter uses float32 on one GPU after float16 corruption was observed on a Kaggle T4. The first run downloads [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) and records its resolved commit. Existing checkpoints resume only with the same source/config/model. Test cases are not evaluated by this command.

In [ ]:
run_directory = Path('/kaggle/working/roho-smoke')
run_args = ['-m', 'roho', 'run', '--config', 'configs/kaggle-smoke.json', '--out', str(run_directory)]
if (run_directory / 'state.json').exists():
    run_args.append('--resume')
command(*run_args)


## Checks
### Evaluate the frozen pilot once
This opens the five-case pilot-test batch after the driver freezes every final harness. Do not tune this run after seeing these outcomes. Predeclared larger experiments require fresh datasets/configuration. A completed batch will not be rerun.

In [ ]:
command('-m', 'roho', 'evaluate', '--out', str(run_directory), '--confirm-frozen')
command('-m', 'roho', 'report', '--out', str(run_directory))
from IPython.display import Markdown, display
display(Markdown((run_directory / 'REPORT.md').read_text()))


### Save complete measurements
Download the ZIP below. It includes source/config hashes, the resolved model revision, prompts/proposals, decisions, task traces, token accounting and final outcomes. Keep it private until author review.

In [ ]:
archive_path = shutil.make_archive('/kaggle/working/roho-smoke-results', 'zip', run_directory)
print('Download:', archive_path)


## Next Steps
Inspect baseline success, invalid-proposal rate and actual GPU usage before attempting `configs/kaggle-pilot.json`. The larger pilot schedules at most 564 task episodes before skips; its shared-template data are not the main paper benchmark. Follow KAGGLE.md for exact commands. Stop the GPU session after saving outputs. No positive outcome is presumed.